In [ ]:
import glob
import math
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm

In [ ]:
# Subset Olink RVAT significant

# olink_burden_test_file = "proteomics_prs_am_loftee_mac20_burden_regression_results.parquet"
# olink_burden_test_file = "proteomics_prs_loftee_mac20_burden_regression_results.parquet"
olink_burden_test_file = "proteomics_prs_df_loftee_mac20_burden_regression_results.parquet"
!dx download project-REDACTED:/processed_data/olink/blacklist/{olink_burden_test_file} -o PATH_TO_FILE

olink_whitelist = (
    pl.read_parquet('PATH_TO_FILE')
    .rename({'gene': 'region'})
    .filter((pl.col('padj')<=0.05) & (pl.col('wilcox_padj')<=0.05) )
    .select(['region'])
    .with_columns(
        phenotype = pl.col('region') + '_olink'
    )
)

# olink_correlations_file = "olink_all_mac20_lofteeHC_correlations.parquet"
# !dx download project-REDACTED:/processed_data/REGENIE_results/{olink_correlations_file} -o PATH_TO_FILE

# olink_corrs = (
#     pl.read_parquet('PATH_TO_FILE')
#     .with_columns(
#         loftee_corr = pl.col('correlation'),
#         loftee_corr_abs = pl.col('correlation').abs(),
#         loftee_corr_dir = pl.col('correlation')/pl.col('correlation').abs(),
#     )
#     .drop_nans()
#     .select(['region', 'phenotype', 'loftee_corr', 'loftee_corr_abs', 'loftee_corr_dir']) 
# )

# olink_whitelist = (
#     olink_whitelist
#     .join(olink_corrs, on=['region', 'phenotype'], how='inner')
#     .drop_nans()
#     # .sort('loftee_corr_abs', descending=True)
#     # .sort('pval_fdr')
#     # .unique(subset=["region"], keep="first", maintain_order=True)
# )

olink_whitelist

In [ ]:
# EUR unrelated individuals

!dx download project-REDACTED:/processed_data/sample_lists/olink_cauc_3rd_degree_samples.csv -o PATH_TO_FILE

unrel_eur_samples = pl.read_csv('PATH_TO_FILE')['sample'].cast(pl.Utf8).to_list()
print(len(unrel_eur_samples))
unrel_eur_samples[:5]

In [ ]:
# prot_file = "cauc_cov_regression_90pcs_prs"  # covariates and PRS corrected
# prot_file = "cauc_protrider_lite_prs_rint"   # PROTRIDER corrected (RINT)
prot_file = "cauc_protrider_lite_prs_t_df"     # PROTRIDER corrected (T distribution)

# Download Olink:
!dx download project-REDACTED:/processed_data/olink/adjusted/{prot_file}.parquet -o PATH_TO_FILE

phenos = pl.read_parquet('PATH_TO_FILE')

olink_genes2use = list(set(phenos.columns).intersection(set(olink_whitelist['region'])))

phenos = phenos.select(['sample'] + olink_genes2use)

long_phenos = (
    phenos
    .unpivot(
        index='sample',
        on=olink_genes2use,
        variable_name='phenotype',
        value_name='pheno_value'
    )

    .with_columns(
        # region = pl.col('phenotype'),
        phenotype = pl.col('phenotype') + '_olink',
    )
    .drop_nulls()
)

print(long_phenos['phenotype'].value_counts(sort=True))
long_phenos

In [ ]:
long_phenos.filter(pl.col('phenotype')=='ENSG00000141642_olink').select(pl.col('pheno_value').std())

In [ ]:
mac = 20

RAP_ANNO_DIR = "project-REDACTED:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated"

LOCAL_DIR = 'PATH_TO_FILE'

# ANNO_FILE = "annotations_fillna_ukbgym.parquet"
ANNO_FILE = "annotations_fillna_ukbgym_with_mane.parquet"

!dx download {RAP_ANNO_DIR}/{ANNO_FILE} -o {LOCAL_DIR}/{ANNO_FILE}

id_list = (
    pl.scan_parquet(f'{LOCAL_DIR}/{ANNO_FILE}')
    .filter(
        pl.col('region').is_in(olink_whitelist.select('region').unique().to_series()),
        pl.col('mac_ukb')<=mac,
    )
    .select('id')
    .unique()
    .collect()
)

id_list

In [ ]:
# Download genotype (long gt) file
!dx download project-REDACTED:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/gt_long.parquet -o PATH_TO_FILE

long_gt = (
    pl.scan_parquet('PATH_TO_FILE')
    .select(['id', 'sample', 'gt'])
    .filter(
        pl.col('gt')==1,
        pl.col('sample').is_in(unrel_eur_samples),
    )
    .join(
        id_list.lazy(),
        on='id',
        how='semi'
    )

    .collect()
)

long_gt

In [ ]:
import math

output_dir = 'PATH_TO_FILE'
!mkdir -p {output_dir}

pheno_list = long_phenos['phenotype'].unique().to_list()
CHUNK_SIZE = 100
num_phenos = len(pheno_list)
num_chunks = math.ceil(num_phenos / CHUNK_SIZE)

# Process in Batches
for i in tqdm(range(0, num_phenos, CHUNK_SIZE)):
    # 1. Define the current batch of genes
    chunk_phenos = pheno_list[i : i + CHUNK_SIZE]

    print(f"Processing chunk starting at index: {i}")
    (
        long_phenos.lazy()
        .filter(pl.col('phenotype').is_in(chunk_phenos))
        .join(
            long_gt.lazy(),
            on='sample',
            how='inner'
        )
        .group_by(['id', 'phenotype'])
        .agg(
            n_individuals = pl.len().cast(pl.Int32),
            mean_pheno_value = pl.col('pheno_value').mean().cast(pl.Float32),
            std_pheno_value = pl.col('pheno_value').std().cast(pl.Float32),
        )
        .with_columns(
            mean_pheno_value_rank=pl.col('mean_pheno_value')
                .rank(descending=True, method="max")      # Olink is always under expression
                .over('phenotype')
                .cast(pl.Float32),
        )
        .with_columns(
            mean_pheno_value_ptile=(
                pl.col('mean_pheno_value_rank') / pl.len().over('phenotype')
            ).cast(pl.Float32),
        )

        .sink_parquet(f'{output_dir}/tmp_appv_chunk_{i}.parquet')
    )


In [ ]:
# Create small appv file directly from chunks — no 92GB intermediate
# Each raw chunk (~8GB) gets filtered to gene-trait associations (~50MB), then combined.
output_dir = 'PATH_TO_FILE'
small_output_file = "PATH_TO_FILE"

# Pre-collect id_region filtered to whitelist regions only (much smaller than full anno)
id_region = (
    pl.scan_parquet(f'{LOCAL_DIR}/{ANNO_FILE}')
    .filter(pl.col('region').is_in(olink_whitelist['region']))
    .select(['id', 'region'])
    .unique()
    .collect()
)
print(f"id_region: {id_region.shape}")

# Filter each raw chunk → gene-trait associations, deduplicate, collect
raw_files = sorted(glob.glob(f'{output_dir}/tmp_appv_chunk_*.parquet'))
filtered_chunks = []

for f in tqdm(raw_files):
    chunk = (
        pl.scan_parquet(f)
        .join(id_region.lazy(), on='id', how='inner')
        .join(olink_whitelist.lazy(), on=['region', 'phenotype'], how='inner')
        .drop('region')
        .unique(subset=['id', 'phenotype'])
        .collect(engine='streaming')
    )
    filtered_chunks.append(chunk)
    print(f"  {f.split('/')[-1]}: {chunk.shape}")

result = pl.concat(filtered_chunks)
print(f"\nCombined: {result.shape}")
result.write_parquet(small_output_file)
print(f"Written to {small_output_file}")

In [ ]:
# Verify small file
pl.scan_parquet(small_output_file).head().collect()

In [ ]:
!dx upload {small_output_file} --path project-REDACTED:/processed_data/ukbgym/avg_pheno_per_var/